
# Evidently Drift Analysis — Databricks Notebook (Propensity Model Retrain)
This notebook compares **Old** vs **New** datasets for a propensity model retrain.  
It is optimized for Spark/Databricks and focuses on **data drift, target drift, and prediction drift**.

**What you get**
- Safe schema alignment (common columns only)
- Robust sampling for large tables
- Evidently `DataDriftPreset` + per-column PSI/KS/Chi2
- Target/prediction drift (if provided)
- A clean drift summary table

> Tip: Run cells top-to-bottom. Adjust paths and parameters in the next cell.


In [ ]:

# ====== PARAMETERS (EDIT HERE) ======

# Input sources (Parquet or Delta path in DBFS / external storage)
OLD_PATH = "/path/to/old_data.parquet"
NEW_PATH = "/path/to/new_data.parquet"

# Column names (set None if not available)
TARGET_COL = "label"        # binary target (0/1) if present
PRED_SCORE_COL = "score"    # predicted probability score if present
PRED_LABEL_COL = None       # predicted class label if present (optional)

# Sampling for large data (fraction 0< f <=1). Use smaller (e.g., 0.1–0.2) for huge tables.
SAMPLE_OLD = 0.15
SAMPLE_NEW = 0.15
SEED_OLD = 42
SEED_NEW = 43

# If table is extremely wide, analyze only top-K columns (set None for all)
TOP_K_NUM = None
TOP_K_CAT = None

# Exclude clearly non-informative columns (ids, timestamps) or known leakage fields
EXCLUDE_COLS = {"member_id", "claim_id", "event_ts", "created_at"}

# Max distinct categories to keep for a categorical column; others will be bucketed (optional future step)
MAX_CATS = 200

# ====== END PARAMETERS ======


In [ ]:

# If running on Databricks, uncomment the next line for a clean install on the cluster:
# %pip install evidently==0.4.34 scikit-multiflow==0.5.3

from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np

from evidently import ColumnMapping
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, TargetDriftPreset, ClassificationPreset
from evidently.metrics import ColumnDriftMetric

print("Libraries loaded.")

In [ ]:

# Load Spark DataFrames
df_old = spark.read.parquet(OLD_PATH)
df_new = spark.read.parquet(NEW_PATH)

print(f"Old rows: {df_old.count():,}, New rows: {df_new.count():,}")
print(f"Old cols: {len(df_old.columns)}, New cols: {len(df_new.columns)}")

# Align columns
old_cols, new_cols = set(df_old.columns), set(df_new.columns)
only_in_old = sorted(list(old_cols - new_cols))
only_in_new = sorted(list(new_cols - old_cols))

print("Columns only in old:", only_in_old[:20], ("... (+more)" if len(only_in_old)>20 else ""))
print("Columns only in new:", only_in_new[:20], ("... (+more)" if len(only_in_new)>20 else ""))

common_cols = sorted(list(old_cols & new_cols))
# Drop excluded
common_cols = [c for c in common_cols if c not in EXCLUDE_COLS]

# Ensure target/pred cols are present or set to None
if TARGET_COL not in common_cols: TARGET_COL = None
if PRED_SCORE_COL not in common_cols: PRED_SCORE_COL = None
if PRED_LABEL_COL not in common_cols: PRED_LABEL_COL = None

df_old = df_old.select(common_cols)
df_new = df_new.select(common_cols)

print(f"Using {len(common_cols)} common columns after exclusions.")


In [ ]:

# Identify numeric & categorical columns from Spark schema
num_types = (IntegerType, LongType, FloatType, DoubleType, ShortType, DecimalType)
cat_types = (StringType, BooleanType)

numeric_cols = [f.name for f in df_old.schema.fields if isinstance(f.dataType, num_types)]
categorical_cols = [f.name for f in df_old.schema.fields if isinstance(f.dataType, cat_types)]

# Remove target/pred from feature lists
for special in [TARGET_COL, PRED_SCORE_COL, PRED_LABEL_COL]:
    if special in numeric_cols: numeric_cols.remove(special)
    if special in categorical_cols: 
        try: categorical_cols.remove(special)
        except ValueError: pass

# Optional narrowing
if TOP_K_NUM is not None:
    numeric_cols = numeric_cols[:TOP_K_NUM]
if TOP_K_CAT is not None:
    categorical_cols = categorical_cols[:TOP_K_CAT]

print(f"Numeric features: {len(numeric_cols)} | Categorical features: {len(categorical_cols)}")


In [ ]:

def null_profile(sdf, cols):
    exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]
    row = sdf.select(*exprs).collect()[0].asDict()
    n = sdf.count()
    return pd.DataFrame({
        "column": list(row.keys()),
        "null_count": list(row.values()),
        "null_pct": [row[c]/max(n,1) for c in row.keys()]
    }).sort_values("null_pct", ascending=False)

null_df_old = null_profile(df_old, common_cols)
null_df_new = null_profile(df_new, common_cols)
display(null_df_old.head(20))
display(null_df_new.head(20))

# Flag heavy-null columns that may distort drift tests
HEAVY_NULL_THRESHOLD = 0.5
heavy_null = set(null_df_old.query("null_pct > @HEAVY_NULL_THRESHOLD")["column"]).union(
             set(null_df_new.query("null_pct > @HEAVY_NULL_THRESHOLD")["column"]))
if heavy_null:
    print("Warning: heavy-null columns detected and will be excluded:", list(heavy_null)[:20])
    numeric_cols = [c for c in numeric_cols if c not in heavy_null]
    categorical_cols = [c for c in categorical_cols if c not in heavy_null]


In [ ]:

# Sample for scalability
old_sample = df_old.sample(withReplacement=False, fraction=float(SAMPLE_OLD), seed=SEED_OLD)
new_sample = df_new.sample(withReplacement=False, fraction=float(SAMPLE_NEW), seed=SEED_NEW)

# Keep only selected columns
selected_cols = numeric_cols + categorical_cols
for c in [TARGET_COL, PRED_SCORE_COL, PRED_LABEL_COL]:
    if c and c not in selected_cols:
        selected_cols.append(c)

old_pd = old_sample.select(selected_cols).toPandas()
new_pd = new_sample.select(selected_cols).toPandas()

# Ensure categorical dtype is 'object' for Evidently
for c in categorical_cols:
    if c in old_pd: old_pd[c] = old_pd[c].astype("object")
    if c in new_pd: new_pd[c] = new_pd[c].astype("object")

print(f"Pandas samples — old: {old_pd.shape}, new: {new_pd.shape}")


In [ ]:

col_map = ColumnMapping(
    target=TARGET_COL,
    prediction=PRED_LABEL_COL if PRED_LABEL_COL else PRED_SCORE_COL,
    numerical_features=numeric_cols,
    categorical_features=categorical_cols,
)

print(col_map)


In [ ]:

# Overall dataset drift
data_drift_report = Report(metrics=[DataDriftPreset()])
data_drift_report.run(reference_data=old_pd, current_data=new_pd, column_mapping=col_map)

# Per-column with specific stattests
metrics = []
for c in numeric_cols:
    metrics.append(ColumnDriftMetric(column_name=c, stattest='psi'))   # PSI for numeric
for c in categorical_cols:
    metrics.append(ColumnDriftMetric(column_name=c, stattest='chi2'))  # Chi-square for categorical

col_drift_report = Report(metrics=metrics)
col_drift_report.run(reference_data=old_pd, current_data=new_pd, column_mapping=col_map)

print("Drift reports computed.")


In [ ]:

if TARGET_COL is not None:
    tgt_report = Report(metrics=[TargetDriftPreset()])
    tgt_report.run(reference_data=old_pd, current_data=new_pd, column_mapping=col_map)
    print("Target drift computed.")
else:
    tgt_report = None

# If you have discrete predictions, you can also compute classification quality drift:
if TARGET_COL is not None and PRED_LABEL_COL is not None:
    clf_report = Report(metrics=[ClassificationPreset()])
    use_cols = [c for c in [TARGET_COL, PRED_LABEL_COL] + numeric_cols + categorical_cols if c]
    clf_report.run(reference_data=old_pd[use_cols], current_data=new_pd[use_cols], column_mapping=col_map)
    print("Classification quality drift computed.")
else:
    clf_report = None


In [ ]:

def drift_table_from_report(rep: Report) -> pd.DataFrame:
    res = rep.as_dict()
    rows = []
    for sec in res.get("metrics", []):
        if "result" in sec and "drift_by_columns" in sec["result"]:
            by_cols = sec["result"]["drift_by_columns"]
            for col, info in by_cols.items():
                rows.append({
                    "column": col,
                    "stattest_name": info.get("stattest_name"),
                    "p_value": info.get("p_value"),
                    "drift_score": info.get("drift_score"),
                    "drift_detected": info.get("drift_detected"),
                })
    return pd.DataFrame(rows).sort_values(["drift_detected", "p_value"], ascending=[False, True])

summary_df = drift_table_from_report(data_drift_report)
display(summary_df)

# Column-level stattest summary
col_summary = drift_table_from_report(col_drift_report)
display(col_summary)


In [ ]:

def safe_rate(series):
    s = pd.Series(series).dropna()
    if s.empty: return np.nan
    return float((s == 1).mean()) if sorted(s.unique()) in [[0,1],[0],[1]] else np.nan

if TARGET_COL:
    old_pos_rate = safe_rate(old_pd[TARGET_COL])
    new_pos_rate = safe_rate(new_pd[TARGET_COL])
    print(f"Positive rate old vs new: {old_pos_rate:.4f} vs {new_pos_rate:.4f}")
else:
    print("TARGET_COL not set; skipping positive-rate comparison.")


In [ ]:

# Optional: Save HTML reports (uncomment if needed)
# data_drift_report.save_html('/dbfs/FileStore/evidently_data_drift.html')
# col_drift_report.save_html('/dbfs/FileStore/evidently_col_drift.html')
# if tgt_report: tgt_report.save_html('/dbfs/FileStore/evidently_target_drift.html')
# if clf_report: clf_report.save_html('/dbfs/FileStore/evidently_clf_quality.html')
# print("Reports saved to /dbfs/FileStore")
